In [10]:
#2021.08.18. WED
#Team_밥믈리에

## NLP
#00. 패키지 호출
import warnings
import pandas as pd 
import numpy as np 
import re 
import csv
from konlpy.tag import Okt

#00-1. 사전변수 정의하기.
CULTIVAR = '고시히카리'
NUM_OF_CHAR = 2
WORD_COUNT_LIMIT = 1

#00-2. warning message ignore
warnings.filterwarnings(action='ignore')


In [11]:
#01. 데이터셋 결합하기. 
#(1) 쿠팡 데이터셋 불러오기. 
review_data_01 = pd.read_excel(f'../data/Coupang/{CULTIVAR}.xlsx')

#(2) 컬럼 이름 변경하기. 
review_data_01.rename(columns={'review' : 'review_01', 'rate' : 'score'}, inplace=True)

#(3) 쿠팡 데이터셋 확인하기.
review_data_01

,name,kg,delivery,score,date,review_01,rice
0,이화에월백하고 고시히까리 3kg 강화섬쌀 ES강화농산 강화쌀 고시히까리 강화섬쌀 강...,3kg,무료배송,5,2021.06.06,NaN,NaN
1,대한농산 보약같은 고시히카리 쌀,5kg,무료배송,5,2021.08.14,NaN,NaN
2,2020년햅쌀/슈퍼오닝/특등급쌀/고시히카리/쌀20Kg/쌀10Kg/쌀4Kg/안중농협쌀...,4kg,무료배송,5,2021.07.21,NaN,NaN
3,2020년햅쌀/슈퍼오닝/특등급쌀/고시히카리/쌀20Kg/쌀10Kg/쌀4Kg/안중농협쌀...,5kg,무료배송,5,2021.06.24,NaN,표면 매끈해요\n도정일 최근이에요\n쌀알 보통이에요
4,2020년햅쌀/슈퍼오닝/특등급쌀/고시히카리/쌀20Kg/쌀10Kg/쌀4Kg/안중농협쌀...,6kg,무료배송,5,2021.05.16,NaN,NaN
...,...,...,...,...,...,...,...
13078,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,4,2016.01.19,NaN,NaN
13079,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,4,2016.01.13,밥이 맛있네요...윤기도 좔좔~~~두번째 구매합니다.번창하십시요,NaN
13080,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,4,2015.12.19,빠른 배송 너무 좋아요,NaN
13081,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,5,2015.11.17,맛있는 김포 고시히카리쌀 잘 받았습니다♥\n배송도 빠르고~아주 만족!,NaN


In [12]:
#(4) 롯데 데이터셋 불러오기.
try              :
    review_data_02 = pd.read_excel('../data/lotteon/crawling_LotteON.xlsx', sheet_name = CULTIVAR)
except Exception :
    review_data_02 = pd.DataFrame({
        'review_01' : [np.nan]
    })
    
#(4) 롯데 데이터셋 확인하기. 
review_data_02 

,product_name,cultivar,review_01,review_02,score,date
0,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,재구매,NO_VALUE,5,1시간 전
1,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,온라인 구입시 품절이 잘되요,NO_VALUE,5,8시간 전
2,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,고시하카리 소량판매 좋아요,NO_VALUE,5,2021.08.17
3,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,잘 받았습니다~~~,NO_VALUE,5,2021.08.16
4,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,한달사용기,NO_VALUE,5,2021.08.14
...,...,...,...,...,...,...
1318,[2020년산] 고시히카리 10kg,고시히카리,당일도정은 아니고 8월6일 도정된거네요. 기존에 먹던게 있는데 요즘 안나와서ㅜ일단 ...,“보통이에요”,5,2020.08.19
1319,윤기가 흐르는 고시히카리쌀 5kg 안성양성농협,고시히카리,고시히까리 인데 별차이모르겠어요\n그저그래요,“보통이에요”,4,2020.07.04
1320,[온세일]2020년 햅쌀 백미/고시히카리/추청쌀 10kg 모음전,고시히카리,마트판매가보다 저렴하게 구입할수 있어 우선 구매했습니다.\n아직 사용전이니 나중에 ...,“재구매의사 있어요”,4,2020.07.13
1321,[2020년산] 고시히카리 20kg,고시히카리,물품 잘 받았습니다. 감사합니다.^^,“재구매의사 있어요”,5,2020.06.15


In [13]:
#(5) 데이터셋 취합하기. 
review_data = pd.concat([review_data_01, review_data_02], axis=0)

#(6) 데이터셋 필요 변수만 추출하기. 
review_data = review_data[['review_01']]

#(7) 취합 데이터셋 확인하기.
review_data

,review_01
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
1318,당일도정은 아니고 8월6일 도정된거네요. 기존에 먹던게 있는데 요즘 안나와서ㅜ일단 ...
1319,고시히까리 인데 별차이모르겠어요\n그저그래요
1320,마트판매가보다 저렴하게 구입할수 있어 우선 구매했습니다.\n아직 사용전이니 나중에 ...
1321,물품 잘 받았습니다. 감사합니다.^^


In [14]:
#02. 데이터셋 전처리하기. 
#(1) 결측값 제거하기. 
review_data[review_data['review_01']=='nan'] = np.nan
review_data = review_data.dropna(axis=0)

#(3) 리스트로 변환하기. 
review_list = review_data['review_01'].values.tolist()

#(4) 데이터셋 구조 확인하기. 
review_list[:5]

['맛있습니다\n맛있습니다 잘먹고있어요',
 '아주맛있다까진아니고 괜찮은편입니다',
 '밥맛이 진짜 조아요~\n밥맛이 진짜 조으네요~',
 '배송 겁내 빠름',
 '밥맛이 정말 좋아요.\n아쉬운 것은 혈당 관리 때문에 잡곡밥을 먹어야 해서...']

In [15]:
#03. 텍스트 전처리하기. 
#(1) 텍스트에서 한글, 영어, 띄어쓰기를 제외하고 모두 삭제하기. 
review_list_pp  = [] 
for i in range(len(review_data)) :
    text = re.sub('[^가-힣 ]', '', review_list[i])
    review_list_pp.append(text)
    
#(2) 처리 결과 확인하기. 
review_list_pp[:5]

['맛있습니다맛있습니다 잘먹고있어요',
 '아주맛있다까진아니고 괜찮은편입니다',
 '밥맛이 진짜 조아요밥맛이 진짜 조으네요',
 '배송 겁내 빠름',
 '밥맛이 정말 좋아요아쉬운 것은 혈당 관리 때문에 잡곡밥을 먹어야 해서']

In [16]:
#04. 자연어처리해 형태소 분석하기.
#(1) okt(twitter) 객체 정의하기
okt = Okt()

#(2) 한 레코드 씩 형태소 분석하기.  
extract_morph_list = []
for i in range (len(review_data)) :
    morph = okt.morphs(review_list_pp[i], stem=True, norm=True)
    extract_morph_list.append(morph)

#(3) 처리 결과 확인하기. 
extract_morph_list[:5]

[['맛있다', '맛있다', '잘', '먹다'],
 ['아주', '맛있다', '아니다', '괜찮다', '편입', '니', '다'],
 ['밥맛', '이', '진짜', '좋다', '밥맛', '이', '진짜', '좋다'],
 ['배송', '겁내', '빠르다'],
 ['밥맛',
  '이',
  '정말',
  '좋다',
  '아쉽다',
  '것',
  '은',
  '혈당',
  '관리',
  '때문',
  '에',
  '잡곡',
  '밥',
  '을',
  '먹다',
  '하다']]

In [17]:
#(4) 한 리스트로 저장하기.
morpheme_list = []
for i in range(len(extract_morph_list)) :
    morpheme_list.extend(extract_morph_list[i])

#(5) 처리 결과 확인하기. 
morpheme_list[:5]

['맛있다', '맛있다', '잘', '먹다', '아주']

In [18]:
#(6) 데이터 프레임 변환하기. 
morpheme_df = pd.DataFrame({'value': morpheme_list})

#(7) 불용어 처리하기. 
STOP_WORD = []
with open('../data/STOPWORD_list.csv', 'r') as file  :
    reader = csv.reader(file)
    for row in reader                                :
        STOP_WORD.append(row)
STOP_WORD = STOP_WORD[0]

for i in range(len(STOP_WORD)) :
    morpheme_df[morpheme_df['value']==STOP_WORD[i]] = np.nan

#(8) 데이터프레임 보기 좋게 구성하기.
word_frequency = morpheme_df.value_counts().to_frame()
word_frequency = word_frequency.reset_index()
word_frequency.columns = ['value','count']
word_frequency

#(9) 단어 글자 갯수가 NUM_OF_CHAR 이하인 값 처리하기
for i in range(len(word_frequency))                  :
    if len(word_frequency['value'][i]) < NUM_OF_CHAR :
        word_frequency.at[i ,'count'] = 0

#(10) count 값이 WORD_COUNT_LIMIT 개 이상인 행만 출력하기. 
word_frequency = word_frequency[word_frequency['count'] >= WORD_COUNT_LIMIT]

#(11) 엑셀로 저장하기. 
word_frequency.to_excel(f'../data/NLP/morph_frequency/{CULTIVAR}.xlsx', index=False)

#(12) 결과 확인하기. 
word_frequency

,value,count
10,윤기,916
33,찰지다,255
76,찰기,106
81,고소하다,97
88,구수하다,86
...,...,...
785,뭉개지,1
787,쫀쫀하,1
790,뭉개져,1
791,씁쓰름하다,1
